# 08 — Chunk size and retrieval limits

A 3,072-dimensional embedding is a vector shape, not a character limit. This experiment holds the source and questions constant while varying the maximum chunk length. It reports full BM25, Gemini cosine, and RRF rankings.

In [1]:
import os
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table
ROOT = bootstrap()
from app.domain.models import Chunk, SearchQuery, SourceLocator
from app.retrieval.dense import GeminiDenseIndex
from app.retrieval.hybrid import HybridRetriever
from ingestion.chunking import chunk_pages
from ingestion.exporters import read_result
from ingestion.indexing import build_runtime_embedding_cache

RUN_GEMINI = os.getenv('RUN_GEMINI_CHUNK_EXPERIMENT', '0') == '1'
SOURCE = ROOT / 'data' / 'processed' / 'foyer_group_qrt_2025'
source_result = read_result(SOURCE)
{'run_gemini': RUN_GEMINI, 'source_pages': len(source_result.pages)}

{'run_gemini': False, 'source_pages': 10}

In [2]:
sizes = (900, 1800, 3000, 4800)
variants = {}
for size in sizes:
    artifacts = chunk_pages(
        source_result.manifest.document_id,
        source_result.manifest.title,
        source_result.pages,
        source_result.manifest.entity,
        source_result.manifest.period,
        max_characters=size,
    )
    variants[size] = [
        Chunk(
            id=artifact.id,
            document_id=artifact.document_id,
            chunk_type=f'experimental_page_text_{size}',
            text=artifact.contextualized_text,
            locator=SourceLocator(
                document_id=artifact.document_id,
                document_title=source_result.manifest.title,
                version=source_result.manifest.sha256[:12],
                source_url=source_result.manifest.source_url,
                page=artifact.page_start,
                section_path=artifact.section_path,
            ),
        )
        for artifact in artifacts
    ]
summary_rows = [
    {
        'max_characters': size,
        'chunks': len(chunks),
        'minimum_characters': min(map(lambda item: len(item.text), chunks)),
        'average_characters': round(sum(map(lambda item: len(item.text), chunks)) / len(chunks), 1),
        'maximum_characters_with_context': max(map(lambda item: len(item.text), chunks)),
    }
    for size, chunks in variants.items()
]
display_table(summary_rows)

,max_characters,chunks,minimum_characters,average_characters,maximum_characters_with_context
0,900,113,94,672.2,987
1,1800,73,103,1052.8,1887
2,3000,56,93,1365.4,3087
3,4800,50,103,1545.8,4887


In [3]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
variant_rows = [
    {
        'max_characters': size,
        'chunk_id': chunk.id,
        'page': chunk.locator.page,
        'characters': len(chunk.text),
        'full_text': chunk.text,
    }
    for size, chunks in variants.items()
    for chunk in chunks
]
display_table(variant_rows)

,max_characters,chunk_id,page,characters,full_text
0,900,foyer_group_qrt_2025-p001-text-000-00,1,115,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Groupe Foyer\nQRT Public 2025
1,900,foyer_group_qrt_2025-p001-text-001-00,1,103,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: S.02.01.02\nBilan
2,900,foyer_group_qrt_2025-p001-text-002-00,1,987,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Valeur\n Solvabilité II\n C0010\nActifs\n Immobilisations incorporelles R0030\n Actifs d’impôts différés R0040\n Excédent du régime de retraite R0050\n Immobilisations corporelles détenues pour usage propre R0060 119.859\n Investissements (autres qu’actifs en représentation de contrats en unités de compte et indexés) R0070 4.241.339\n Bi
3,900,foyer_group_qrt_2025-p001-text-002-01,1,984,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: ens immobiliers (autres que détenus pour usage propre) R0080 108.248\n Détentions dans des entreprises liées, y compris participations R0090 108.114\n Actions R0100 608.154\n Actions - cotées R0110 583.250\n Actions - non cotées R0120 24.904\n Obligations R0130 2.713.529\n Obligations d’État R0140 949.952"
4,900,foyer_group_qrt_2025-p001-text-002-02,1,978,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Obligations d’entreprise R0150 1.232.926\n Titres structurés R0160 530.651\n Titres garantis R0170 0\n Organismes de placement collectif R0180 640.456\n Produits dérivés R0190 10.985\n Dépôts autres que les équivalents de trésorerie R0200 21.397\n Autres investissements R0210 30.455
5,900,foyer_group_qrt_2025-p001-text-002-03,1,983,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Actifs en représentation de contrats en unités de compte et indexés R0220 20.948.552\n Prêts et prêts hypothécaires R0230 48.935\n Avances sur police R0240 631\n Prêts et prêts hypothécaires aux particuliers R0250 17.432\n Autres prêts et prêts hypothécaires R0260 30.872\n Montants recouvrables au titre des contrats de réassurance R0270 61.719\n Non-vie et santé similaire à la non-vie R028
6,900,foyer_group_qrt_2025-p001-text-002-04,1,916,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: 0 60.293\n Non-vie hors santé R0290 60.728\n Santé similaire à la non-vie R0300 -436\n Vie et santé similaire à la vie, hors santé, UC et indexés R0310 5.080\n Santé similaire à la vie R0320 0\n Vie hors santé, UC et indexés R0330 5.080\n Vie UC et indexés R0340 -3.654\n Dépôts auprès des cédantes"
7,900,foyer_group_qrt_2025-p001-text-002-05,1,964,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: R0350 12\n Créances nées d’opérations d’assurance et montants à recevoir d’intermédiaires R0360 144.504\n Créances nées d’opérations de réassurance R0370 1.731\n Autres créances (hors assurance) R0380 68.565\n Actions propres auto-détenues (directement) R0390 18.594\n Éléments de fonds propres ou fonds initial appelé(s), mais non encore payé(s) R0400 0\n Trésorerie et équivalents de trésorerie R0410 568.126\n Autres actifs non mentionnés dans les postes ci-dessus"
8,900,foyer_group_qrt_2025-p001-text-002-06,1,239,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: R0420 4.342\n Total de l’actif R0500 26.226.277
9,900,foyer_group_qrt_2025-p002-text-000-00,2,115,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Groupe Foyer\nQRT Public 2025


In [4]:
test_queries = [
    ('natural', "What public evidence describes Groupe Foyer's prudential coverage in 2025?"),
    ('regulatory_labels', 'Eligible own funds, group SCR and SCR coverage ratio'),
    ('row_codes', 'S.23.01.22 R0660 R0680 R0690 C0010'),
    ('underspecified', 'Foyer solvency'),
]
ranking_rows = []
for size, chunks in variants.items():
    index_directory = ROOT / 'data' / 'processed' / 'chunk_size_experiment' / str(size)
    if RUN_GEMINI:
        build_runtime_embedding_cache(chunks, index_directory)
    if not (index_directory / 'embedding_manifest.json').exists():
        continue
    dense_index = GeminiDenseIndex(chunks, index_directory)
    retriever = HybridRetriever(chunks, dense_index=dense_index)
    for case_id, text in test_queries:
        query = SearchQuery(
            field_id='diagnostic', field_label='Diagnostic', text=text,
            document_ids=[source_result.manifest.document_id],
        )
        for final_rank, candidate in enumerate(
            retriever.search(query, k=len(chunks)), start=1
        ):
            ranking_rows.append({
                'max_characters': size,
                'case': case_id,
                'query': text,
                'final_rank': final_rank,
                'chunk_id': candidate.chunk.id,
                'page': candidate.chunk.locator.page,
                'bm25_rank': candidate.score.lexical_rank,
                'bm25_score': candidate.score.lexical_score,
                'gemini_dense_rank': candidate.score.dense_rank,
                'gemini_cosine_score': candidate.score.dense_score,
                'rrf_score': candidate.score.rrf_score,
                'contains_target_codes': any(code in candidate.chunk.text for code in ('R0660', 'R0680', 'R0690')),
                'full_text': candidate.chunk.text,
            })
if RUN_GEMINI:
    assert ranking_rows
display_table(ranking_rows)

,max_characters,case,query,final_rank,chunk_id,page,bm25_rank,bm25_score,gemini_dense_rank,gemini_cosine_score,rrf_score,contains_target_codes,full_text
0,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,1,foyer_group_qrt_2025-p009-text-001-00,9,12,2.081251,13,0.796075,0.027588,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: S.25.01.22\nCapital de solvabilité requis - pour les groupes qui utilisent la formule standard
1,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,2,foyer_group_qrt_2025-p001-text-001-00,1,2,2.271896,32,0.791543,0.026999,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: S.02.01.02\nBilan
2,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,3,foyer_group_qrt_2025-p002-text-001-00,2,3,2.271896,33,0.791543,0.026626,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: S.02.01.02\nBilan
3,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,4,foyer_group_qrt_2025-p010-text-003-12,10,38,0.029965,1,0.817244,0.026598,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: solvabilité du s’applique
4,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,5,foyer_group_qrt_2025-p003-text-003-01,3,39,0.029625,7,0.799483,0.025026,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: assurance Assurance de de Pertes
5,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,6,foyer_group_qrt_2025-p001-text-000-00,1,22,0.033313,18,0.793567,0.025016,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Groupe Foyer\nQRT Public 2025
6,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,7,foyer_group_qrt_2025-p002-text-000-00,2,23,0.033313,19,0.793567,0.024706,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Groupe Foyer\nQRT Public 2025
7,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,8,foyer_group_qrt_2025-p010-text-003-02,10,35,0.029965,11,0.796892,0.024611,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: pour proportionnelle Date de
8,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,9,foyer_group_qrt_2025-p003-text-000-00,3,24,0.033313,20,0.793567,0.024405,False,Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Groupe Foyer\nQRT Public 2025
9,900,natural,What public evidence describes Groupe Foyer's prudential coverage in 2025?,10,foyer_group_qrt_2025-p003-text-003-02,3,44,0.029293,8,0.799113,0.024321,False,"Document: QRT public 2025 - Groupe Foyer. Entity: Groupe Foyer. Period: 2025. Content: Total Assurance maritime, incendie"


## Interpretation guardrails

Larger chunks may preserve wide tables or complete arguments, but can dilute the signal, increase embedding and generation costs, and return oversized citations. Smaller chunks improve locality but can separate labels from values. The selected size must therefore be justified by target-evidence rank and citation usability, not by embedding dimensionality.